## Data Beyond IID

### Graphs

In [1]:
import dgl.function as fn

FileNotFoundError: Could not find module 'c:\Users\fanyu\miniconda3\envs\graphML\lib\site-packages\dgl\dgl.dll' (or one of its dependencies). Try using the full path with constructor syntax.

In [1]:
import torch
from torch import nn

In [2]:
def gcn_message(edges):
  # The argument is a batch of edges using source node’s feature ‘h’
  return {'msg': edges.src['h']}

def gcn_reduce(nodes):
  # The argument is a batch of nodes with features summed over ‘msg’
  return {'h' : torch.sum(nodes.mailbox['msg'], dim=1)}

class GCNLayer(nn.Module):
  def __init__(self, in_feats, out_feats):
    super(GCNLayer, self).__init__()
    self.linear = nn.Linear(in_feats, out_feats)
    
  def forward(self, g, inputs):
    g.ndata['h'] = inputs
    g.send(g.edges(), gcn_message) # trigger message passing
    g.recv(g.nodes(), gcn_reduce) # trigger aggregation at all nodes
    h = g.ndata.pop('h')          # get the result node features
    return self.linear(h)

In [3]:
class GCN(nn.Module):
  def __init__(self, in_feats, hidden_size, num_classes):
    super(GCN, self).__init__()
    self.gcn1 = GCNLayer(in_feats, hidden_size)
    self.gcn2 = GCNLayer(hidden_size, num_classes)
    
  def forward(self, g, inputs):
    h = self.gcn1(g, inputs)
    h = torch.relu(h)
    h = self.gcn2(g, h)
    return h

# First layer - 34 inputs to 5 dimensions
# Second layer - 2 dimensional embedding for 2 groups
net = GCN(34, 5, 2)
  

In [4]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
all_logits = []

for epoch in range(30):
  logits = net(G, inputs)
  # save the logits for visualization later
  all_logits.append(logits.detach())
  logp = F.log_softmax(logits, 1)
  # only compute loss for labeled nodes
  loss = F.nll_loss(logp[labeled_nodes], labels)
  
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

NameError: name 'G' is not defined